# LeetCode 240 — Search a 2D Matrix II bằng Python

Link đề: https://leetcode.com/problems/search-a-2d-matrix-ii/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search, Divide and Conquer, Matrix

Skeleton LeetCode:

```python
class Solution:
    def searchMatrix(self, matrix: List[List[int]], target: int) -> bool:
        ...
```

Ghi chú: đề, ví dụ và constraints lấy từ đề gốc; mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với một lời giải brute force độc lập (`any(target in row for row in matrix)`) trên ma trận ngẫu nhiên có hàng và cột đều được sắp.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: ma trận `m × n` có **mỗi hàng tăng dần từ trái sang phải** và **mỗi cột tăng dần từ trên xuống dưới**; kiểm tra `target` có trong ma trận không.
2. Nhận ra ma trận này **khác** bài "Search a 2D Matrix" (LeetCode 74): ở đây **không** thể nối các hàng thành một dãy tăng, nên không thể tìm nhị phân một lần trên cả ma trận.
3. Đi từ **duyệt hết `O(m·n)`** → **tìm nhị phân trên từng hàng `O(m log n)`** → **đi theo "bậc thang" từ góc trên-phải `O(m + n)`** (cách nộp bài) → **chia để trị chia ma trận thành bốn phần**.
4. Hiểu **vì sao góc trên-phải (hoặc dưới-trái) là điểm xuất phát đặc biệt**: tại đó, nếu giá trị lớn hơn `target` ta **loại cả một cột**, nếu nhỏ hơn ta **loại cả một hàng** — hai lựa chọn không nhập nhằng.
5. Tránh các lỗi kinh điển: dùng tìm nhị phân trên "ma trận dàn phẳng", xuất phát ở góc trên-trái, quên cột `0` (`col > 0` thay vì `col >= 0`), và đi sai hướng.

## 2. Đề bài & ví dụ

Viết thuật toán hiệu quả để tìm giá trị `target` trong ma trận số nguyên `m × n` `matrix`. Ma trận có các tính chất:

- các số trong **mỗi hàng** được sắp **tăng dần** từ trái sang phải;
- các số trong **mỗi cột** được sắp **tăng dần** từ trên xuống dưới.

Trả về `True` nếu `target` có trong ma trận, ngược lại `False`.

**Ví dụ 1**

- Input: `matrix = [[1,4,7,11,15],[2,5,8,12,19],[3,6,9,16,22],[10,13,14,17,24],[18,21,23,26,30]]`, `target = 5`
- Output: `true`

**Ví dụ 2**

- Input: cùng `matrix`, `target = 20`
- Output: `false`

**Ràng buộc:** `m == matrix.length`; `n == matrix[i].length`; `1 <= n, m <= 300`; `-10⁹ <= matrix[i][j] <= 10⁹`; các số trong mỗi hàng và mỗi cột đều được sắp tăng dần; `-10⁹ <= target <= 10⁹`.

In [1]:
from typing import List
import random
from bisect import bisect_left

MATRIX = [
    [1, 4, 7, 11, 15],
    [2, 5, 8, 12, 19],
    [3, 6, 9, 16, 22],
    [10, 13, 14, 17, 24],
    [18, 21, 23, 26, 30],
]
examples = [
    (MATRIX, 5, True),
    (MATRIX, 20, False),
]
for matrix, target, expected in examples:
    print(f"matrix {len(matrix)}x{len(matrix[0])}, target={target}, kỳ vọng {expected}")
for row in MATRIX:
    print("  ", row)

matrix 5x5, target=5, kỳ vọng True
matrix 5x5, target=20, kỳ vọng False
   [1, 4, 7, 11, 15]
   [2, 5, 8, 12, 19]
   [3, 6, 9, 16, 22]
   [10, 13, 14, 17, 24]
   [18, 21, 23, 26, 30]


## 3. Kiến thức nền

### 3.1. Ma trận là list của list; kích thước `m` và `n`

`matrix[i][j]` là phần tử hàng `i`, cột `j` (đếm từ `0`). `m = len(matrix)` là số hàng, `n = len(matrix[0])` là số cột. Góc trên-phải là `matrix[0][n - 1]`, góc dưới-trái là `matrix[m - 1][0]`.

In [2]:
m, n = len(MATRIX), len(MATRIX[0])
print("m =", m, ", n =", n)
print("góc trên-trái :", MATRIX[0][0])
print("góc trên-phải :", MATRIX[0][n - 1])
print("góc dưới-trái :", MATRIX[m - 1][0])
print("góc dưới-phải :", MATRIX[m - 1][n - 1])

m = 5 , n = 5
góc trên-trái : 1
góc trên-phải : 15
góc dưới-trái : 18
góc dưới-phải : 30


### 3.2. Hai tính chất của ma trận và "đường chéo" khó chịu

Mỗi hàng tăng và mỗi cột tăng, nhưng **không** có quan hệ giữa phần tử cuối của hàng này với phần tử đầu của hàng kế tiếp: hàng `0` kết thúc ở `15`, hàng `1` bắt đầu ở `2` (nhỏ hơn). Vậy nếu dàn phẳng ma trận theo hàng ta **không** được một dãy tăng.

In [3]:
flat = [x for row in MATRIX for x in row]
print(flat)
print("dàn phẳng có tăng dần?", flat == sorted(flat))          # False
print("hàng 0 kết thúc:", MATRIX[0][-1], "| hàng 1 bắt đầu:", MATRIX[1][0])

[1, 4, 7, 11, 15, 2, 5, 8, 12, 19, 3, 6, 9, 16, 22, 10, 13, 14, 17, 24, 18, 21, 23, 26, 30]
dàn phẳng có tăng dần? False
hàng 0 kết thúc: 15 | hàng 1 bắt đầu: 2


### 3.3. Tìm nhị phân trên một dãy đã sắp và `bisect_left`

Với dãy `a` tăng dần, tìm nhị phân cho kết quả trong `O(log n)`. `bisect_left(a, x)` trả vị trí đầu tiên `i` mà `a[i] >= x`; `x` có trong `a` khi `i < len(a)` và `a[i] == x`.

In [4]:
row = [1, 4, 7, 11, 15]
for x in [0, 1, 7, 8, 15, 16]:
    i = bisect_left(row, x)
    print(f"x={x}: bisect_left={i}, có trong hàng? {i < len(row) and row[i] == x}")


def binary_search(row, x):
    """Tự viết tìm nhị phân (không dùng bisect)."""
    lo, hi = 0, len(row) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if row[mid] == x:
            return True
        if row[mid] < x:
            lo = mid + 1
        else:
            hi = mid - 1
    return False


assert all(binary_search(row, x) == (x in row) for x in range(-2, 20))

x=0: bisect_left=0, có trong hàng? False
x=1: bisect_left=0, có trong hàng? True
x=7: bisect_left=2, có trong hàng? True
x=8: bisect_left=3, có trong hàng? False
x=15: bisect_left=4, có trong hàng? True
x=16: bisect_left=5, có trong hàng? False


### 3.4. Ý tưởng "loại bỏ" bằng một phép so sánh

Ở góc trên-phải `v = matrix[0][n-1]` (lớn nhất của hàng `0`, nhỏ nhất của cột `n−1`):

- nếu `v > target`: `v` là nhỏ nhất của cột `n−1`, mọi phần tử cột đó đều `≥ v > target`: **loại cả cột** `n−1`;
- nếu `v < target`: `v` là lớn nhất của hàng `0`, mọi phần tử hàng đó đều `≤ v < target`: **loại cả hàng** `0`.

Mỗi phép so sánh loại một hàng hoặc một cột, nên tối đa `m + n` bước.

In [5]:
r, c = 0, len(MATRIX[0]) - 1
target = 5
v = MATRIX[r][c]
print(f"góc trên-phải = {v}")
print("v > target ->", v > target, "-> loại cả cột", c, ":", [row[c] for row in MATRIX])

góc trên-phải = 15
v > target -> True -> loại cả cột 4 : [15, 19, 22, 24, 30]


### 3.5. Đệ quy trên vùng con của ma trận

Ta có thể mô tả một **vùng con** bằng bốn số `(r1, c1, r2, c2)`: các hàng `r1..r2`, cột `c1..c2` (bao gồm). Hàm đệ quy `search(r1, c1, r2, c2)` kiểm tra vùng đó; vùng rỗng (`r1 > r2` hoặc `c1 > c2`) thì trả `False`.

In [6]:
def region(matrix, r1, c1, r2, c2):
    return [row[c1:c2 + 1] for row in matrix[r1:r2 + 1]]


print(region(MATRIX, 1, 1, 3, 3))      # hàng 1..3, cột 1..3
print(region(MATRIX, 2, 0, 1, 4))      # vùng rỗng: r1 > r2

[[5, 8, 12], [6, 9, 16], [13, 14, 17]]
[]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Tìm `target = 5` trong ma trận ví dụ. Hãy xuất phát ở **góc trên-phải** (`15`):

| bước | ô `(hàng, cột)` | giá trị | so với `5` | hành động |
|---|---|---|---|---|
| 1 | `(0, 4)` | `15` | `15 > 5` | loại cột 4, sang trái |
| 2 | `(0, 3)` | `11` | `11 > 5` | loại cột 3, sang trái |
| 3 | `(0, 2)` | `7` | `7 > 5` | loại cột 2, sang trái |
| 4 | `(0, 1)` | `4` | `4 < 5` | loại hàng 0, xuống dưới |
| 5 | `(1, 1)` | `5` | `5 == 5` | **tìm thấy** |

Với `target = 20` (không có): ta cứ loại hàng hoặc cột cho tới khi ra khỏi ma trận. Mỗi bước hoặc đi sang trái hoặc đi xuống, nên tối đa `m + n − 1` bước.

Điểm quan trọng: **mỗi bước chỉ có một lựa chọn đúng**. Nếu xuất phát ở góc trên-trái (`1`, nhỏ nhất) và `1 < target` thì cả đi sang phải lẫn đi xuống đều có thể đúng: nhập nhằng.

In [7]:
def staircase_trace_quick(matrix, target):
    r, c = 0, len(matrix[0]) - 1
    path = []
    while r < len(matrix) and c >= 0:
        v = matrix[r][c]
        path.append((r, c, v))
        if v == target:
            return True, path
        if v > target:
            c -= 1
        else:
            r += 1
    return False, path


print(staircase_trace_quick(MATRIX, 5))
print(staircase_trace_quick(MATRIX, 20))

(True, [(0, 4, 15), (0, 3, 11), (0, 2, 7), (0, 1, 4), (1, 1, 5)])
(False, [(0, 4, 15), (1, 4, 19), (2, 4, 22), (2, 3, 16), (3, 3, 17), (4, 3, 26), (4, 2, 23), (4, 1, 21), (4, 0, 18)])


## 5. Cách 1 — Brute Force / baseline: duyệt mọi ô

**Ý tưởng.** Hai vòng lặp qua mọi hàng và cột; gặp `target` thì trả `True`. Không dùng tính chất sắp xếp.

In [8]:
def search_matrix_bruteforce(matrix: list[list[int]], target: int) -> bool:
    for row in matrix:
        for x in row:
            if x == target:
                return True
    return False


for matrix, target, expected in examples:
    print(f"target={target} -> {search_matrix_bruteforce(matrix, target)} (kỳ vọng {expected})")

target=5 -> True (kỳ vọng True)
target=20 -> False (kỳ vọng False)


**Độ phức tạp (Cách 1).** Gọi `m` là số hàng, `n` là số cột. **Time `O(m·n)`**, **Space `O(1)`**. Với `300 × 300 = 9·10⁴` ô vẫn nhanh, nhưng không dùng tính chất đề cho.

**Nút thắt.** Ta bỏ phí thông tin "hàng và cột đã sắp". Mỗi hàng đã sắp cho phép tìm nhị phân; còn hơn thế, ta có thể loại cả hàng hoặc cột bằng một phép so sánh.

## 6. Ý tưởng thuật toán / bước đột phá

**Bậc thang (staircase) từ góc trên-phải.** Đặt `(r, c) = (0, n − 1)`. Lặp khi `r < m` và `c ≥ 0`:

- `matrix[r][c] == target`: tìm thấy, trả `True`;
- `matrix[r][c] > target`: giá trị là **nhỏ nhất của cột `c`** (vì cột tăng từ trên xuống, và ta đang ở hàng trên cùng còn lại) nên cả cột `c` còn lại đều `> target`: `c -= 1`;
- `matrix[r][c] < target`: giá trị là **lớn nhất của hàng `r`** (vì hàng tăng từ trái sang, ta đang ở cột phải nhất còn lại) nên cả hàng `r` còn lại đều `< target`: `r += 1`.

Mỗi bước loại một hàng hoặc một cột, nên tổng số bước `≤ m + n`.

Đối xứng: có thể xuất phát từ **góc dưới-trái** (`r = m − 1`, `c = 0`): `v > target` thì `r -= 1`, `v < target` thì `c += 1`. Hai góc còn lại (trên-trái, dưới-phải) **không** hoạt động vì từ đó cả hai hướng đều tăng hoặc đều giảm, nên không biết nên loại bên nào.

In [9]:
def staircase_demo(matrix, target, corner="top-right"):
    m, n = len(matrix), len(matrix[0])
    r, c = (0, n - 1) if corner == "top-right" else (m - 1, 0)
    steps = 0
    while 0 <= r < m and 0 <= c < n:
        steps += 1
        v = matrix[r][c]
        if v == target:
            return True, steps
        if corner == "top-right":
            if v > target:
                c -= 1
            else:
                r += 1
        else:
            if v > target:
                r -= 1
            else:
                c += 1
    return False, steps


for t in (5, 20, 18, 30):
    print(f"target={t}: trên-phải -> {staircase_demo(MATRIX, t)}, dưới-trái -> {staircase_demo(MATRIX, t, 'bottom-left')}")

target=5: trên-phải -> (True, 5), dưới-trái -> (True, 5)
target=20: trên-phải -> (False, 9), dưới-trái -> (False, 8)
target=18: trên-phải -> (True, 9), dưới-trái -> (True, 1)
target=30: trên-phải -> (True, 5), dưới-trái -> (True, 5)


## 7. Các cách tiếp theo

### Cách 2 — Tìm nhị phân trên từng hàng (cách của lời giải tham khảo)

**Giải thích.** Mỗi hàng đã sắp, nên với mỗi hàng dùng `bisect_left` để tìm phần tử đầu tiên `>= target` và kiểm tra có bằng `target` không. Không dùng tính chất cột.

In [10]:
def search_matrix_row_bisect(matrix: list[list[int]], target: int) -> bool:
    for row in matrix:
        j = bisect_left(row, target)             # O(log n) trên mỗi hàng
        if j < len(row) and row[j] == target:
            return True
    return False


for matrix, target, expected in examples:
    print(f"target={target} -> {search_matrix_row_bisect(matrix, target)} (kỳ vọng {expected})")

target=5 -> True (kỳ vọng True)
target=20 -> False (kỳ vọng False)


**Độ phức tạp Cách 2:** **Time `O(m log n)`**, **Space `O(1)`**. Tốt hơn Cách 1 khi `m` nhỏ, nhưng chưa dùng tính chất cột.

### Cách 3 — Bậc thang từ góc trên-phải (cách nộp bài)

**Giải thích.** Cài đặt đúng mục 6: `r = 0`, `c = n − 1`; trong khi còn trong ma trận so sánh `matrix[r][c]` với `target` và đi sang trái/xuống.

In [11]:
def search_matrix_staircase(matrix: list[list[int]], target: int) -> bool:
    m, n = len(matrix), len(matrix[0])
    r, c = 0, n - 1                               # góc trên-phải
    while r < m and c >= 0:
        v = matrix[r][c]
        if v == target:
            return True
        if v > target:
            c -= 1                                # loại cả cột c
        else:
            r += 1                                # loại cả hàng r
    return False


for matrix, target, expected in examples:
    print(f"target={target} -> {search_matrix_staircase(matrix, target)} (kỳ vọng {expected})")

target=5 -> True (kỳ vọng True)
target=20 -> False (kỳ vọng False)


In [12]:
# Code nộp LeetCode cho Cách 3
from typing import List


class Solution:
    def searchMatrix(self, matrix: List[List[int]], target: int) -> bool:
        m, n = len(matrix), len(matrix[0])
        r, c = 0, n - 1
        while r < m and c >= 0:
            if matrix[r][c] == target:
                return True
            if matrix[r][c] > target:
                c -= 1
            else:
                r += 1
        return False

**Độ phức tạp Cách 3:** **Time `O(m + n)`** (mỗi bước loại một hàng hoặc một cột; tối đa `m + n − 1` bước), **Space `O(1)`**.

### Cách 4 — Chia để trị: chia vùng thành bốn phần quanh phần tử giữa

**Giải thích.** Xét vùng `(r1, c1, r2, c2)`. Lấy phần tử giữa `v = matrix[mr][mc]` với `mr, mc` là trung điểm. Nếu `v == target` thì xong.

- Nếu `v < target`: mọi phần tử ở **góc trên-trái** (hàng `≤ mr`, cột `≤ mc`) đều `≤ v < target`: **loại** phần này; tìm ở ba phần còn lại (trên-phải, dưới-trái, dưới-phải).
- Nếu `v > target`: mọi phần tử ở **góc dưới-phải** (hàng `≥ mr`, cột `≥ mc`) đều `≥ v > target`: loại phần này; tìm ở ba phần còn lại (trên-trái, trên-phải, dưới-trái).

Mỗi lần loại bớt một phần tư vùng.

In [13]:
def search_matrix_divide(matrix: list[list[int]], target: int) -> bool:
    def search(r1, c1, r2, c2):
        if r1 > r2 or c1 > c2:                          # vùng rỗng
            return False
        mr, mc = (r1 + r2) // 2, (c1 + c2) // 2
        v = matrix[mr][mc]
        if v == target:
            return True
        if v < target:
            # loại góc trên-trái (hàng r1..mr, cột c1..mc)
            return (search(r1, mc + 1, mr, c2)          # trên-phải
                    or search(mr + 1, c1, r2, mc)       # dưới-trái
                    or search(mr + 1, mc + 1, r2, c2))  # dưới-phải
        # v > target: loại góc dưới-phải (hàng mr..r2, cột mc..c2)
        return (search(r1, c1, mr - 1, mc - 1)          # trên-trái
                or search(r1, mc, mr - 1, c2)           # trên-phải
                or search(mr, c1, r2, mc - 1))          # dưới-trái

    return search(0, 0, len(matrix) - 1, len(matrix[0]) - 1)


for matrix, target, expected in examples:
    print(f"target={target} -> {search_matrix_divide(matrix, target)} (kỳ vọng {expected})")

target=5 -> True (kỳ vọng True)
target=20 -> False (kỳ vọng False)


**Độ phức tạp Cách 4:** gọi `N = m·n`; mỗi lần đệ quy tạo tối đa 3 vùng con kích thước ≈ `N/4`: `T(N) = 3T(N/4) + O(1)` nên **Time `O(N^{log₄ 3}) ≈ O(N^{0.79})`** (với ma trận vuông `n × n` là `O(n^{1.58})`), kém hơn Cách 3 (`O(m + n)`). **Space `O(log(m·n))`** cho ngăn xếp đệ quy. Vẫn nhanh hơn duyệt `O(m·n)` và minh họa tư duy chia để trị.

> Cách 3 là cách nộp bài tốt nhất. Cách 2 dễ viết. Cách 4 cho cách nhìn chia để trị. Cách 1 là chuẩn đối chiếu.

## 8. TRACE từng bước

Ta theo dõi Cách 3 (bậc thang) với `target = 5` (có), `target = 20` (không), và `target = 18` (nằm ở góc dưới-trái, đi hết cột `0`).

In [14]:
def trace_staircase(matrix, target):
    m, n = len(matrix), len(matrix[0])
    r, c = 0, n - 1
    print(f"target={target}; xuất phát ở góc trên-phải (0, {n - 1})")
    step = 0
    while r < m and c >= 0:
        step += 1
        v = matrix[r][c]
        if v == target:
            print(f"  bước {step}: ({r},{c}) = {v} == {target} -> TÌM THẤY")
            return True
        if v > target:
            print(f"  bước {step}: ({r},{c}) = {v} > {target} -> loại cột {c}, c = {c - 1}")
            c -= 1
        else:
            print(f"  bước {step}: ({r},{c}) = {v} < {target} -> loại hàng {r}, r = {r + 1}")
            r += 1
    print(f"  ra khỏi ma trận (r={r}, c={c}) -> KHÔNG CÓ")
    return False


assert trace_staircase(MATRIX, 5) is True
print()
assert trace_staircase(MATRIX, 20) is False
print()
assert trace_staircase(MATRIX, 18) is True

target=5; xuất phát ở góc trên-phải (0, 4)
  bước 1: (0,4) = 15 > 5 -> loại cột 4, c = 3
  bước 2: (0,3) = 11 > 5 -> loại cột 3, c = 2
  bước 3: (0,2) = 7 > 5 -> loại cột 2, c = 1
  bước 4: (0,1) = 4 < 5 -> loại hàng 0, r = 1
  bước 5: (1,1) = 5 == 5 -> TÌM THẤY

target=20; xuất phát ở góc trên-phải (0, 4)
  bước 1: (0,4) = 15 < 20 -> loại hàng 0, r = 1
  bước 2: (1,4) = 19 < 20 -> loại hàng 1, r = 2
  bước 3: (2,4) = 22 > 20 -> loại cột 4, c = 3
  bước 4: (2,3) = 16 < 20 -> loại hàng 2, r = 3
  bước 5: (3,3) = 17 < 20 -> loại hàng 3, r = 4
  bước 6: (4,3) = 26 > 20 -> loại cột 3, c = 2
  bước 7: (4,2) = 23 > 20 -> loại cột 2, c = 1
  bước 8: (4,1) = 21 > 20 -> loại cột 1, c = 0
  bước 9: (4,0) = 18 < 20 -> loại hàng 4, r = 5
  ra khỏi ma trận (r=5, c=0) -> KHÔNG CÓ

target=18; xuất phát ở góc trên-phải (0, 4)
  bước 1: (0,4) = 15 < 18 -> loại hàng 0, r = 1
  bước 2: (1,4) = 19 > 18 -> loại cột 4, c = 3
  bước 3: (1,3) = 12 < 18 -> loại hàng 1, r = 2
  bước 4: (2,3) = 16 < 18 -> loại h

Bảng tóm tắt (`target = 20`):

| bước | `(r, c)` | giá trị | so sánh | hành động |
|---|---|---|---|---|
| 1 | `(0, 4)` | 15 | `< 20` | `r = 1` |
| 2 | `(1, 4)` | 19 | `< 20` | `r = 2` |
| 3 | `(2, 4)` | 22 | `> 20` | `c = 3` |
| 4 | `(2, 3)` | 16 | `< 20` | `r = 3` |
| 5 | `(3, 3)` | 17 | `< 20` | `r = 4` |
| 6 | `(4, 3)` | 26 | `> 20` | `c = 2` |
| 7 | `(4, 2)` | 23 | `> 20` | `c = 1` |
| 8 | `(4, 1)` | 21 | `> 20` | `c = 0` |
| 9 | `(4, 0)` | 18 | `< 20` | `r = 5` (ra khỏi ma trận) |

Không có `20`. Tổng `9 ≤ m + n − 1 = 9` bước.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (bậc thang từ góc trên-phải). Đúng class `Solution`, đúng method `searchMatrix(self, matrix: List[List[int]], target: int) -> bool`.

In [15]:
from typing import List


class Solution:
    def searchMatrix(self, matrix: List[List[int]], target: int) -> bool:
        m, n = len(matrix), len(matrix[0])
        r, c = 0, n - 1                      # xuất phát ở góc trên-phải
        while r < m and c >= 0:              # còn nằm trong ma trận (cột 0 vẫn hợp lệ: c >= 0)
            value = matrix[r][c]
            if value == target:
                return True
            if value > target:               # cả cột c phía dưới đều lớn hơn target
                c -= 1
            else:                            # cả hàng r phía trái đều nhỏ hơn target
                r += 1
        return False


for matrix, target, expected in examples:
    assert Solution().searchMatrix(matrix, target) == expected
print("PASS ví dụ đề cho class Solution")

PASS ví dụ đề cho class Solution


## 10. Vì sao thuật toán đúng

- **Bất biến:** nếu `target` có trong ma trận thì nó nằm trong **vùng còn lại** gồm các hàng `r..m−1` và các cột `0..c`. Ban đầu vùng là cả ma trận.
- **Ở ô `(r, c)` (góc trên-phải của vùng còn lại):**
  - `v > target`: các phần tử cột `c` trong vùng (hàng `r..m−1`) đều `≥ v` (cột tăng dần xuống dưới), nên `> target`; không chứa `target`. Bỏ cột `c`: vùng còn lại vẫn chứa `target` (nếu có).
  - `v < target`: các phần tử hàng `r` trong vùng (cột `0..c`) đều `≤ v` (hàng tăng từ trái sang), nên `< target`. Bỏ hàng `r`: bất biến giữ.
- **Kết thúc:** mỗi bước giảm `c` hoặc tăng `r` nên dừng sau tối đa `m + n − 1` bước; khi vùng rỗng, `target` không có.
- **Vì sao góc trên-trái không dùng được:** tại đó phần tử nhỏ nhất của vùng; nếu `v < target` thì `target` có thể nằm bên phải hoặc bên dưới, không loại được hàng hay cột nào.
- **Cách 2:** đúng vì mỗi hàng được tìm nhị phân đầy đủ. **Cách 4:** phần bị loại luôn là phần mà mọi phần tử `≤ v < target` hoặc `≥ v > target` nhờ hai tính chất sắp xếp.

## 11. Phân tích Time / Space Complexity

Gọi `m` là số hàng, `n` là số cột.

- **Cách 1 (duyệt hết):** Time `O(m·n)`, Space `O(1)`.
- **Cách 2 (nhị phân từng hàng):** Time `O(m log n)`, Space `O(1)`.
- **Cách 3 (bậc thang):** Time **`O(m + n)`**, Space `O(1)`.
- **Cách 4 (chia để trị):** Time `O((m·n)^{log₄ 3})`, Space `O(log(m·n))`.
- Trường hợp xấu nhất của Cách 3: `target` không có hoặc ở góc dưới-trái, mất `m + n − 1` bước; tốt nhất: `1` bước. Với `m = n = 300`: tối đa `599` bước.

## 12. Bộ test

Ta gom các cách vào từ điển `impls`. `oracle` là `any(target in row for row in matrix)`. Ma trận ngẫu nhiên được sinh sao cho **mỗi hàng và mỗi cột đều không giảm** (có thể có giá trị trùng): `a[i][j] = max(a[i−1][j], a[i][j−1]) + gia_tăng` với `gia_tăng ≥ 0`, nên các tính chất của đề thỏa (đề dùng "sắp tăng dần" nhưng ta thử cả trường hợp có giá trị trùng để kiểm tra độ bền).

In [16]:
sol = Solution()
impls = {
    "bruteforce": search_matrix_bruteforce,
    "row_bisect": search_matrix_row_bisect,
    "staircase": search_matrix_staircase,
    "divide": search_matrix_divide,
    "submit": sol.searchMatrix,
}


def oracle(matrix, target):
    return any(target in row for row in matrix)


def random_matrix(m, n, max_step=3, base=None):
    """Ma trận m x n có hàng và cột không giảm."""
    a = [[0] * n for _ in range(m)]
    start = random.randint(-20, 20) if base is None else base
    for i in range(m):
        for j in range(n):
            up = a[i - 1][j] if i else start
            left = a[i][j - 1] if j else start
            a[i][j] = max(up, left) + random.randint(0, max_step)
    return a


def check_properties(matrix):
    return (all(row[j] <= row[j + 1] for row in matrix for j in range(len(row) - 1))
            and all(matrix[i][j] <= matrix[i + 1][j] for i in range(len(matrix) - 1) for j in range(len(matrix[0]))))


def run_all(matrix, target):
    return {name: fn(matrix, target) for name, fn in impls.items()}


for matrix, target, expected in examples:
    assert oracle(matrix, target) == expected
    for name, got in run_all(matrix, target).items():
        assert got == expected, (name, target, got, expected)
print("PASS ví dụ của đề (mọi cách và oracle độc lập)")

PASS ví dụ của đề (mọi cách và oracle độc lập)


### 12.1. Các trường hợp biên

In [17]:
def check_all(matrix, target, expected, label):
    assert check_properties(matrix), (label, "ma trận không thỏa đề")
    assert oracle(matrix, target) == expected, (label, "oracle")
    for name, got in run_all(matrix, target).items():
        assert got == expected, (label, name, got, expected)
    print("PASS", label, "->", expected)


check_all([[5]], 5, True, "1 x 1, có")
check_all([[5]], 4, False, "1 x 1, không có (nhỏ hơn)")
check_all([[5]], 6, False, "1 x 1, không có (lớn hơn)")
check_all([[1, 3, 5, 7]], 7, True, "một hàng, phần tử cuối")
check_all([[1, 3, 5, 7]], 1, True, "một hàng, phần tử đầu")
check_all([[1], [3], [5], [7]], 5, True, "một cột")
check_all([[1], [3], [5], [7]], 4, False, "một cột, không có")
check_all(MATRIX, 1, True, "góc trên-trái (cột 0)")
check_all(MATRIX, 30, True, "góc dưới-phải")
check_all(MATRIX, 15, True, "góc trên-phải")
check_all(MATRIX, 18, True, "góc dưới-trái (cột 0, hàng cuối)")
check_all(MATRIX, 0, False, "nhỏ hơn mọi phần tử")
check_all(MATRIX, 31, False, "lớn hơn mọi phần tử")
check_all(MATRIX, 12, True, "phần tử nằm giữa")
check_all(MATRIX, 13, True, "phần tử hàng 3")
check_all([[-10**9, 10**9]], -10**9, True, "giá trị biên -10^9")
check_all([[-10**9, 10**9]], 10**9, True, "giá trị biên 10^9")
check_all([[2, 2, 2], [2, 2, 2]], 2, True, "toàn giá trị trùng, có")
check_all([[2, 2, 2], [2, 2, 2]], 3, False, "toàn giá trị trùng, không có")
big = random_matrix(300, 300)
check_all(big, big[299][0], True, "300 x 300, góc dưới-trái")
check_all(big, big[0][299], True, "300 x 300, góc trên-phải")

PASS 1 x 1, có -> True
PASS 1 x 1, không có (nhỏ hơn) -> False
PASS 1 x 1, không có (lớn hơn) -> False
PASS một hàng, phần tử cuối -> True
PASS một hàng, phần tử đầu -> True
PASS một cột -> True
PASS một cột, không có -> False
PASS góc trên-trái (cột 0) -> True
PASS góc dưới-phải -> True
PASS góc trên-phải -> True
PASS góc dưới-trái (cột 0, hàng cuối) -> True
PASS nhỏ hơn mọi phần tử -> False
PASS lớn hơn mọi phần tử -> False
PASS phần tử nằm giữa -> True
PASS phần tử hàng 3 -> True
PASS giá trị biên -10^9 -> True
PASS giá trị biên 10^9 -> True
PASS toàn giá trị trùng, có -> True
PASS toàn giá trị trùng, không có -> False
PASS 300 x 300, góc dưới-trái -> True
PASS 300 x 300, góc trên-phải -> True


### 12.2. Tính chất và phản ví dụ

In [18]:
random.seed(240)

# Tính chất 1: mọi phần tử của ma trận đều tìm thấy; với mọi giá trị không có thì False
for _ in range(100):
    m, n = random.randint(1, 8), random.randint(1, 8)
    matrix = random_matrix(m, n)
    present = {x for row in matrix for x in row}
    for x in present:
        assert sol.searchMatrix(matrix, x) is True
    for x in range(min(present) - 3, max(present) + 4):
        assert sol.searchMatrix(matrix, x) == (x in present)
print("PASS tính chất: tìm thấy mọi phần tử có mặt và không tìm thấy phần tử vắng mặt")

# Tính chất 2: số bước của bậc thang <= m + n - 1
def staircase_steps(matrix, target):
    m, n = len(matrix), len(matrix[0])
    r, c, steps = 0, n - 1, 0
    while r < m and c >= 0:
        steps += 1
        v = matrix[r][c]
        if v == target:
            return steps
        if v > target:
            c -= 1
        else:
            r += 1
    return steps

for _ in range(300):
    m, n = random.randint(1, 20), random.randint(1, 20)
    matrix = random_matrix(m, n)
    assert staircase_steps(matrix, random.randint(-30, 100)) <= m + n - 1
print("PASS tính chất: bậc thang dùng tối đa m + n - 1 bước")

# Phản ví dụ: tìm nhị phân trên ma trận dàn phẳng (cách của bài 74) sai với ma trận này
def sai_dan_phang(matrix, target):
    flat = [x for row in matrix for x in row]
    i = bisect_left(flat, target)
    return i < len(flat) and flat[i] == target

print("target=5 -> sai (dàn phẳng):", sai_dan_phang(MATRIX, 5), "| đúng:", sol.searchMatrix(MATRIX, 5))
assert sai_dan_phang(MATRIX, 5) is False and sol.searchMatrix(MATRIX, 5) is True

PASS tính chất: tìm thấy mọi phần tử có mặt và không tìm thấy phần tử vắng mặt
PASS tính chất: bậc thang dùng tối đa m + n - 1 bước
target=5 -> sai (dàn phẳng): False | đúng: True


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

In [19]:
random.seed(2026)
for trial in range(3000):
    m, n = random.randint(1, 10), random.randint(1, 10)
    matrix = random_matrix(m, n, max_step=random.choice([0, 1, 3, 10]))
    assert check_properties(matrix)
    lo = min(matrix[0][0], -5)
    hi = matrix[-1][-1] + 5
    target = random.choice([random.randint(lo, hi), random.choice(random.choice(matrix))])
    expected = oracle(matrix, target)
    for name, got in run_all(matrix, target).items():
        assert got == expected, (name, matrix, target, got, expected)

for trial in range(60):                     # ma trận lớn (tới 300 x 300)
    m, n = random.randint(100, 300), random.randint(100, 300)
    matrix = random_matrix(m, n, max_step=random.choice([1, 3, 50]))
    target = random.choice([random.randint(matrix[0][0] - 3, matrix[-1][-1] + 3), random.choice(random.choice(matrix))])
    expected = oracle(matrix, target)
    for name, got in run_all(matrix, target).items():
        assert got == expected, (name, m, n, target)
print("PASS 3060 ma trận ngẫu nhiên: mọi cách khớp oracle độc lập")

PASS 3060 ma trận ngẫu nhiên: mọi cách khớp oracle độc lập


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dàn phẳng ma trận rồi tìm nhị phân (nhầm với bài 74)

In [20]:
def sai_tim_nhi_phan_dan_phang(matrix, target):
    flat = [x for row in matrix for x in row]
    lo, hi = 0, len(flat) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if flat[mid] == target:
            return True
        if flat[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return False


print("target=5 -> sai:", sai_tim_nhi_phan_dan_phang(MATRIX, 5), "| đúng:", Solution().searchMatrix(MATRIX, 5))
print("target=12 -> sai:", sai_tim_nhi_phan_dan_phang(MATRIX, 12), "| đúng:", Solution().searchMatrix(MATRIX, 12))

target=5 -> sai: True | đúng: True
target=12 -> sai: False | đúng: True


**Vì sao sai:** ở bài 74 mỗi hàng bắt đầu lớn hơn hàng trước, nên dàn phẳng là dãy tăng. Ở bài này hàng `1` bắt đầu bằng `2` nhỏ hơn cuối hàng `0` (`15`), nên dàn phẳng **không sắp**, tìm nhị phân cho kết quả sai (bỏ sót `5` và `12`).

### Lỗi 2: Xuất phát ở góc trên-trái

In [21]:
def sai_goc_tren_trai(matrix, target):
    r, c = 0, 0
    while r < len(matrix) and c < len(matrix[0]):
        v = matrix[r][c]
        if v == target:
            return True
        if v < target:
            c += 1                          # nhưng có thể đáng lẽ phải đi xuống
        else:
            return False
    return False


print("target=5 -> sai:", sai_goc_tren_trai(MATRIX, 5), "| đúng:", Solution().searchMatrix(MATRIX, 5))

target=5 -> sai: False | đúng: True


**Vì sao sai:** ở góc trên-trái, nếu `v < target` thì `target` có thể nằm sang phải hoặc xuống dưới: **không loại được** hàng hay cột nào, nên mọi lựa chọn cố định đều bỏ sót. Phải chọn góc mà hai hướng "trái" và "xuống" **giảm / tăng ngược nhau**: góc trên-phải hoặc dưới-trái.

### Lỗi 3: Điều kiện dừng `c > 0` làm bỏ qua cột `0`

In [22]:
def sai_bo_cot_0(matrix, target):
    r, c = 0, len(matrix[0]) - 1
    while r < len(matrix) and c > 0:                 # sai: phải là c >= 0
        v = matrix[r][c]
        if v == target:
            return True
        if v > target:
            c -= 1
        else:
            r += 1
    return False


print("target=18 -> sai:", sai_bo_cot_0(MATRIX, 18), "| đúng:", Solution().searchMatrix(MATRIX, 18))
print("target=1  -> sai:", sai_bo_cot_0(MATRIX, 1), "| đúng:", Solution().searchMatrix(MATRIX, 1))

target=18 -> sai: False | đúng: True
target=1  -> sai: False | đúng: True


**Vì sao sai:** chỉ số cột hợp lệ là `0..n−1`, nên điều kiện còn trong ma trận là `c >= 0`. `c > 0` thoát vòng lặp khi `c = 0`, không bao giờ xét cột `0` (như `1` và `18`).

### Lỗi 4: Đi sai hướng (đảo `r` và `c`)

In [23]:
def sai_huong(matrix, target):
    r, c = 0, len(matrix[0]) - 1
    while r < len(matrix) and c >= 0:
        v = matrix[r][c]
        if v == target:
            return True
        if v > target:
            r += 1                              # sai: giá trị lớn hơn thì phải loại CỘT
        else:
            c -= 1
    return False


print("target=5 -> sai:", sai_huong(MATRIX, 5), "| đúng:", Solution().searchMatrix(MATRIX, 5))

target=5 -> sai: False | đúng: True


**Vì sao sai:** ở góc trên-phải, `v > target` nghĩa là `v` nhỏ nhất của cột, nên cả cột lớn hơn `target` và ta phải **sang trái** (`c -= 1`). Đi xuống thì gặp các giá trị còn lớn hơn: bỏ sót. Nhớ: **lớn hơn thì lùi cột, nhỏ hơn thì tăng hàng**.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Ma trận có **hàng và cột đều sắp** (nhưng không nối thành một dãy): dùng **bậc thang từ góc trên-phải hoặc dưới-trái** `O(m + n)`.
- Ma trận mà mỗi hàng bắt đầu lớn hơn cuối hàng trước (bài 74): tìm nhị phân trên dãy dàn phẳng `O(log(m·n))`.
- Chỉ hàng sắp: tìm nhị phân từng hàng `O(m log n)`.
- Chọn điểm xuất phát sao cho **mỗi so sánh loại bỏ được một hàng hoặc một cột**.
- Ngoài việc tìm có/không, bậc thang còn **đếm số phần tử `≤ x`** trong `O(m + n)` (dùng cho bài "Kth Smallest Element in a Sorted Matrix").
- Cẩn thận chỉ số `0` ở biên: dùng `c >= 0` (hoặc dùng `while True` kèm kiểm tra rõ ràng).

## 15. Biến thể của bài

1. **Search a 2D Matrix (LeetCode 74):** mỗi hàng bắt đầu lớn hơn cuối hàng trước → tìm nhị phân dàn phẳng.
2. **Trả về vị trí `(hàng, cột)`** của `target` (hoặc `(-1, -1)`).
3. **Đếm số phần tử `≤ x`** (và từ đó tìm phần tử nhỏ thứ `k`, LeetCode 378).
4. **Đếm số phần tử âm** trong ma trận sắp (LeetCode 1351).
5. **Ma trận chỉ sắp theo hàng:** `O(m log n)`.

In [24]:
def search_matrix_74(matrix, target):
    """Biến thể 1: hàng sau bắt đầu lớn hơn cuối hàng trước -> tìm nhị phân trên dãy dàn phẳng ảo."""
    m, n = len(matrix), len(matrix[0])
    lo, hi = 0, m * n - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        v = matrix[mid // n][mid % n]          # ánh xạ chỉ số phẳng -> (hàng, cột)
        if v == target:
            return True
        if v < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return False


def find_position(matrix, target):
    """Biến thể 2."""
    m, n = len(matrix), len(matrix[0])
    r, c = 0, n - 1
    while r < m and c >= 0:
        v = matrix[r][c]
        if v == target:
            return (r, c)
        if v > target:
            c -= 1
        else:
            r += 1
    return (-1, -1)


def count_less_equal(matrix, x):
    """Biến thể 3: số phần tử <= x bằng bậc thang từ góc dưới-trái."""
    m, n = len(matrix), len(matrix[0])
    r, c, count = m - 1, 0, 0
    while r >= 0 and c < n:
        if matrix[r][c] <= x:
            count += r + 1                       # cả cột c từ hàng 0..r đều <= x
            c += 1
        else:
            r -= 1
    return count


def count_negatives(matrix):
    """Biến thể 4: số phần tử âm (hàng và cột sắp tăng dần)."""
    return count_less_equal(matrix, -1)


# kiểm tra
assert search_matrix_74([[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]], 3) is True
assert search_matrix_74([[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]], 13) is False
assert find_position(MATRIX, 5) == (1, 1) and find_position(MATRIX, 20) == (-1, -1)
random.seed(15)
for _ in range(200):
    m, n = random.randint(1, 8), random.randint(1, 8)
    matrix = random_matrix(m, n)
    x = random.randint(-30, 60)
    assert count_less_equal(matrix, x) == sum(v <= x for row in matrix for v in row)
    assert count_negatives(matrix) == sum(v < 0 for row in matrix for v in row)
    pos = find_position(matrix, x)
    assert (pos != (-1, -1)) == oracle(matrix, x)
    if pos != (-1, -1):
        assert matrix[pos[0]][pos[1]] == x
print("PASS biến thể: bài 74, vị trí, đếm <= x, đếm số âm")

PASS biến thể: bài 74, vị trí, đếm <= x, đếm số âm


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt hết | hai vòng lặp | `O(m·n)` | `O(1)` | hiểu đề; oracle |
| 2. Nhị phân từng hàng | `bisect_left` trên mỗi hàng | `O(m log n)` | `O(1)` | khi chỉ hàng sắp / `m` nhỏ |
| 3. Bậc thang góc trên-phải | `v > target` thì lùi cột, `v < target` thì tăng hàng | **`O(m + n)`** | `O(1)` | **cách nộp bài** |
| 4. Chia để trị | loại một phần tư theo phần tử giữa | `O((m·n)^0.79)` | `O(log(m·n))` | minh họa chia để trị |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `bisect_left([1, 3, 5], 4)`, `len(MATRIX), len(MATRIX[0])`, `MATRIX[2][1]`, `Solution().searchMatrix([[1, 2], [3, 4]], 4)`, `Solution().searchMatrix([[1, 2], [3, 4]], 5)`.

**Bài 2 (tính tay).** Với ma trận `[[1,3,5],[2,4,6],[7,8,9]]` và `target = 4`: liệt kê từng bước của bậc thang (ô, giá trị, hành động). Làm lại với `target = 7` và `target = 10`.

**Bài 3 (điền TODO).** Hoàn thành `search_todo(matrix, target)` theo Cách 3 (bậc thang).

In [25]:
def search_todo(matrix, target):
    m, n = len(matrix), len(matrix[0])
    r, c = 0, n - 1
    # TODO 1: while r < m and c >= 0:
    #   TODO 2: nếu matrix[r][c] == target trả True
    #   TODO 3: nếu lớn hơn target thì c -= 1; ngược lại r += 1
    # TODO 4: hết vòng lặp trả False
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `search_bottom_left(matrix, target)`: bậc thang xuất phát từ **góc dưới-trái**. Kiểm tra bằng hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `kth_smallest_sorted_matrix(matrix, k)` trả về phần tử nhỏ thứ `k` (đếm từ `1`) của ma trận (hàng và cột sắp tăng) bằng **tìm nhị phân trên giá trị** kết hợp `count_less_equal`. (b) Viết `count_occurrences(matrix, target)` đếm số lần `target` xuất hiện.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `bisect_left` cho vị trí chèn bên trái; chỉ số bắt đầu từ `0`. Bài 2: từ góc trên-phải, lớn hơn `target` thì sang trái, nhỏ hơn thì xuống.
- **Mức 2 (vừa):** Bài 1: `2`, `(5, 5)`, `6`, `True`, `False`. Bài 2: `[[1,3,5],[2,4,6],[7,8,9]]`, `target = 4`: `(0,2)=5>4 → c=1`; `(0,1)=3<4 → r=1`; `(1,1)=4` tìm thấy. `target = 7`: `5<7 → r=1`; `6<7 → r=2`; `(2,2)=9>7 → c=1`; `(2,1)=8>7 → c=0`; `(2,0)=7` thấy. `target = 10`: `5<10 → r=1`, `6<10 → r=2`, `9<10 → r=3` ra khỏi: không có. Bài 3: bốn TODO trong comment.
- **Mức 3 (sâu):** Bài 4: `r = m - 1`, `c = 0`; `v > target` thì `r -= 1`, `v < target` thì `c += 1`. Bài 5(a): tìm nhị phân `x` trong `[matrix[0][0], matrix[-1][-1]]` sao cho `count_less_equal(x) >= k`. Bài 5(b): `count_less_equal(target) − count_less_equal(target − 1)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [26]:
# Bài 1
assert bisect_left([1, 3, 5], 4) == 2
assert (len(MATRIX), len(MATRIX[0])) == (5, 5)
assert MATRIX[2][1] == 6
assert Solution().searchMatrix([[1, 2], [3, 4]], 4) is True
assert Solution().searchMatrix([[1, 2], [3, 4]], 5) is False

# Bài 2
M2 = [[1, 3, 5], [2, 4, 6], [7, 8, 9]]
assert find_position(M2, 4) == (1, 1) and find_position(M2, 7) == (2, 0) and find_position(M2, 10) == (-1, -1)
assert staircase_trace_quick(M2, 4)[1] == [(0, 2, 5), (0, 1, 3), (1, 1, 4)]

# Bài 3
def search_todo_ans(matrix, target):
    m, n = len(matrix), len(matrix[0])
    r, c = 0, n - 1
    while r < m and c >= 0:
        if matrix[r][c] == target:
            return True
        if matrix[r][c] > target:
            c -= 1
        else:
            r += 1
    return False

for matrix, target, expected in examples:
    assert search_todo_ans(matrix, target) == expected

# Bài 4
def search_bottom_left(matrix, target):
    m, n = len(matrix), len(matrix[0])
    r, c = m - 1, 0
    while r >= 0 and c < n:
        v = matrix[r][c]
        if v == target:
            return True
        if v > target:
            r -= 1
        else:
            c += 1
    return False

for matrix, target, expected in examples:
    assert search_bottom_left(matrix, target) == expected

# Bài 5(a), 5(b)
def kth_smallest_sorted_matrix(matrix, k):
    lo, hi = matrix[0][0], matrix[-1][-1]
    while lo < hi:
        mid = (lo + hi) // 2
        if count_less_equal(matrix, mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

def count_occurrences(matrix, target):
    return count_less_equal(matrix, target) - count_less_equal(matrix, target - 1)

random.seed(19)
for _ in range(200):
    m, n = random.randint(1, 7), random.randint(1, 7)
    matrix = random_matrix(m, n)
    flat = sorted(v for row in matrix for v in row)
    k = random.randint(1, m * n)
    assert kth_smallest_sorted_matrix(matrix, k) == flat[k - 1]
    x = random.choice(flat)
    assert count_occurrences(matrix, x) == flat.count(x)
    assert search_bottom_left(matrix, x) is True
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (bậc thang từ góc trên-phải)

```python
class Solution:
    def searchMatrix(self, matrix: List[List[int]], target: int) -> bool:
        m, n = len(matrix), len(matrix[0])
        r, c = 0, n - 1
        while r < m and c >= 0:
            if matrix[r][c] == target:
                return True
            if matrix[r][c] > target:
                c -= 1
            else:
                r += 1
        return False
```

**Checklist**

- [ ] Xuất phát ở **góc trên-phải** (hoặc dưới-trái), không phải trên-trái?
- [ ] `v > target` thì **lùi cột** (`c -= 1`); `v < target` thì **tăng hàng** (`r += 1`)?
- [ ] Điều kiện còn trong ma trận: `r < m and c >= 0` (cột `0` vẫn hợp lệ)?
- [ ] Không nhầm với bài 74 (không được tìm nhị phân trên ma trận dàn phẳng)?
- [ ] Đã thử: `1 × 1`, một hàng, một cột, `target` nhỏ/lớn hơn mọi phần tử, góc dưới-trái, giá trị trùng?
- [ ] Đã đối chiếu với `any(target in row for row in matrix)` trên ma trận ngẫu nhiên?
- [ ] Skeleton đúng `class Solution` / `def searchMatrix(self, matrix: List[List[int]], target: int) -> bool`?